# Optimised Whole-Body Human Generation — StyleGAN-Human + InsetGAN + ReStyle

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdebanjiAdelowo/Whole-body-GAN-generator/blob/main/server/Copy_of_Optimised_User_Whole_Body_Generation.ipynb)

End-to-end pipeline for **face-guided whole-body image generation** with a REST API:

```
Input face photo (URL or upload)
       │
       ▼
  Face alignment (dlib 68-point landmarks)
       │
       ▼
  ReStyle pSp encoder  ──►  face latent codes (iterative, 5 steps)
       │
       ▼
  InsetGAN dual optimiser  ──►  face + body joint refinement
       │
       ▼
  Output: full-body PNG + MP4 video sequence
       │
       ▼
  FastAPI server  ──►  public URL via ngrok
```

**Three-part structure**

| Part | Description |
|---|---|
| **1 — StyleGAN-Human** | Clone repo, download pretrained body generator |
| **2 — ReStyle Encoder** | Clone encoder, download pSp checkpoint, run inversion |
| **3 — API** | FastAPI endpoints for joint optimisation + single image generation |

**Before running**
- Enable GPU: *Runtime → Change runtime type → GPU*
- Set your ngrok token in the server cell (get one free at [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken))


In [1]:
!nvidia-smi

Thu May 21 22:41:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   61C    P0             29W /   70W |    1099MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Environment Verification & API Dependencies

Check GPU availability, then install all required packages.  
**After the pip install, restart the runtime if prompted** — then run all cells from the top.


In [2]:
# After this cell completes, restart the runtime if prompted,
# then re-run all cells from the top.
!pip install fastapi uvicorn pyngrok lpips python-multipart nest_asyncio -q


## Part 1 — StyleGAN-Human: Repository & Pretrained Models

Clones [Lakshmanaraja/User-Whole-Body-Generation](https://github.com/Lakshmanaraja/User-Whole-Body-Generation)  
and downloads the pretrained weights into `StyleGAN_Human/pretrained_models/`.

| Model file | Purpose |
|---|---|
| `stylegan2_1024.pkl` | Body generator (StyleGAN2, 1024 px portrait) |
| `ffhq.pkl` | FFHQ face generator — decoder reference |
| `mmod_human_face_detector.dat` | dlib CNN face detector |
| `shape_predictor_68_face_landmarks.dat` | dlib 68-point landmark predictor |


In [3]:
import os
if not os.path.exists('/content/User-Whole-Body-Generation'):
    !git clone https://github.com/Lakshmanaraja/User-Whole-Body-Generation.git
else:
    print('Repo already exists, skipping clone.')


Repo already exists, skipping clone.


In [4]:
import os, subprocess

repo_path     = '/content/User-Whole-Body-Generation'
stylegan_path = os.path.join(repo_path, 'StyleGAN_Human')

if not os.path.exists(repo_path):
    subprocess.run(['git', 'clone',
        'https://github.com/Lakshmanaraja/User-Whole-Body-Generation.git',
        repo_path], check=True)

if not os.path.exists(stylegan_path):
    raise FileNotFoundError(
        f'StyleGAN_Human not found. Repo contents: {os.listdir(repo_path)}')

os.chdir(stylegan_path)
print('Working directory:', os.getcwd())


Working directory: /content/User-Whole-Body-Generation/StyleGAN_Human


### Model Download Helper

`get_download_model_command()` builds a `wget` one-liner that handles  
Google Drive's confirm-token redirect (older large-file downloads).  
Every model file is saved into `<repo>/StyleGAN_Human/pretrained_models/`  
so all downstream code shares a single consistent path.


In [5]:
def get_download_model_command(file_id, file_name):
    """ Get wget download command for downloading the desired model and save to directory ../pretrained_models. """
    current_directory = os.getcwd()
    save_path = os.path.join(os.path.dirname(current_directory), f'{repo_name}',"pretrained_models")
    if not os.path.exists(save_path):
        os.makedirs(save_path)
    url = r"""wget --load-cookies /tmp/cookies.txt "https://docs.google.com/uc?export=download&confirm=$(wget --quiet --save-cookies /tmp/cookies.txt --keep-session-cookies --no-check-certificate 'https://docs.google.com/uc?export=download&id={FILE_ID}' -O- | sed -rn 's/.*confirm=([0-9A-Za-z_]+).*/\1\n/p')&id={FILE_ID}" -O {SAVE_PATH}/{FILE_NAME} && rm -rf /tmp/cookies.txt""".format(FILE_ID=file_id, FILE_NAME=file_name, SAVE_PATH=save_path)
    return url

In [6]:
experiment_type = 'stylegan2_1024' 
repo_name = 'StyleGAN_Human'\
#version= 2
MODEL_PATHS = {
    "stylegan2_1024": {"id": "1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5", "name": "stylegan2_1024.pkl"},
} 
path = MODEL_PATHS[experiment_type]
download_command = get_download_model_command(file_id=path["id"], file_name=path["name"])
!{download_command}

--2026-05-21 22:41:23--  https://docs.google.com/uc?export=download&confirm=&id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5
Resolving docs.google.com (docs.google.com)... 142.250.101.138, 142.250.101.100, 142.250.101.102, ...
Connecting to docs.google.com (docs.google.com)|142.250.101.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&export=download [following]
--2026-05-21 22:41:23--  https://drive.usercontent.google.com/download?id=1FlAb1rYa0r_--Zj_ML8e6shmaF28hQb5&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.101.132, 2607:f8b0:4023:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.101.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2437 (2.4K) [text/html]
Saving to: ‘/content/User-Whole-Body-Generation/StyleGAN_Human/pretrained_models/stylegan2_1024.pkl’


In [7]:
## Download pretrained StyleGAN on FFHQ 1024x1024 and dlib dat.
ffhq_ckpt = get_download_model_command(file_id="125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M", file_name='ffhq.pkl')
dlib_detector = get_download_model_command(file_id="1MduBgju5KFNrQfDLoQXJ_1_h5MnctCIG", file_name='mmod_human_face_detector.dat')
dlib_landmark = get_download_model_command(file_id="1A82DnJBJzt8wI2J8ZrCK5fgHcQ2-tcWM", file_name='shape_predictor_68_face_landmarks.dat')
!{ffhq_ckpt}
!{dlib_detector}
!{dlib_landmark}

--2026-05-21 22:41:24--  https://docs.google.com/uc?export=download&confirm=&id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M
Resolving docs.google.com (docs.google.com)... 142.250.101.138, 142.250.101.100, 142.250.101.102, ...
Connecting to docs.google.com (docs.google.com)|142.250.101.138|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M&export=download [following]
--2026-05-21 22:41:24--  https://drive.usercontent.google.com/download?id=125OG7SMkXI-Kf2aqiwLLHyCvSW-gZk3M&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 142.250.101.132, 2607:f8b0:4023:c06::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|142.250.101.132|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-05-21 22:41:24 ERROR 404: Not Found.

--2026-05-21 22:41:26--  https://docs.google.com/uc?export=download&confirm=&id=1Md

## Part 2 — ReStyle Encoder: Inversion Pipeline

Encodes a face photo into the StyleGAN latent space using the ReStyle pSp encoder  
([pixel2Style2pixel](https://github.com/eladrich/pixel2style2pixel)).  
The latent codes are then passed to the InsetGAN dual optimiser.

**Key compatibility fix:** The `pSp` decoder is monkey-patched to filter out  
shape-mismatched checkpoint tensors. The FFHQ checkpoint has square noise buffers  
(e.g. `1024×1024`) but the whole-body generator uses portrait buffers (`1024×512`).  
See the patch cell below for details.


### Reset Working Directory

Return to `/content` before cloning external repos so all paths stay consistent.

In [8]:
os.chdir('/content')

### ReStyle Encoder Repository

Clones a modified fork of the [ReStyle encoder](https://github.com/Lakshmanaraja/restyle) whose `utils/` folder has been adapted for StyleGAN-Human alignment.

In [9]:
if not os.path.exists('/content/restyle'):
    !git clone https://github.com/Lakshmanaraja/restyle.git
else:
    print('Restyle repo already exists, skipping clone.')


Restyle repo already exists, skipping clone.


### Python Dependencies

- **lpips** — Learned Perceptual Image Patch Similarity loss, used during optimisation  
- **Ninja** — fast C++ build system required by StyleGAN's custom CUDA ops

In [10]:
!pip install lpips

In [11]:
!wget -q https://github.com/ninja-build/ninja/releases/download/v1.8.2/ninja-linux.zip
!sudo unzip -o ninja-linux.zip -d /usr/local/bin/
!sudo update-alternatives --install /usr/bin/ninja ninja /usr/local/bin/ninja 1 --force


Archive:  ninja-linux.zip
  inflating: /usr/local/bin/ninja    


The `-o` flag on `unzip` forces overwrite without prompting, so re-running this cell will not block on an interactive yes/no question.

### Downloader & Output Directories

`download_with_pydrive = False` — uses `gdown` so no Google OAuth is needed.  
Output directories (`/content/models`, `/content/output/`) are created here.

In [12]:
import os

pretrained_model_dir = os.path.join('/content', 'models')
os.makedirs(pretrained_model_dir, exist_ok=True)

restyle_dir = os.path.join('/content', 'restyle')

output_dir       = os.path.join('/content', 'output')
output_model_dir = os.path.join(output_dir, 'models')
output_image_dir = os.path.join(output_dir, 'images')

download_with_pydrive = False  # uses gdown — no OAuth required for public Drive files

class Downloader(object):
    def __init__(self, use_pydrive):
        self.use_pydrive = use_pydrive
        if self.use_pydrive:
            self.authenticate()

    def authenticate(self):
        from pydrive.auth import GoogleAuth
        from pydrive.drive import GoogleDrive
        from google.colab import auth
        from oauth2client.client import GoogleCredentials
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        self.drive = GoogleDrive(gauth)

    def download_file(self, file_id, file_dst):
        if self.use_pydrive:
            downloaded = self.drive.CreateFile({'id': file_id})
            downloaded.FetchMetadata(fetch_all=True)
            downloaded.GetContentFile(file_dst)
        else:
            os.system(f'gdown --id {file_id} -O "{file_dst}"')

downloader = Downloader(download_with_pydrive)
print('pretrained_model_dir:', pretrained_model_dir)
print('restyle_dir:', restyle_dir)


pretrained_model_dir: /content/models
restyle_dir: /content/restyle


### Core Imports

Standard ML stack plus ReStyle utilities.  
`sys.path.append(restyle_dir)` makes the cloned encoder importable.

In [13]:
from argparse import Namespace

import sys
import numpy as np

from PIL import Image

import torch
import torchvision.transforms as transforms

sys.path.append(restyle_dir)

device = 'cuda'


### StyleGAN Weight Conversion Paths

These variables point to the StyleGAN-Human directory.  
They are used by the `.pkl → .pt` conversion script if the pre-converted file is not already present.

In [14]:
# Paths for optional .pkl → .pt weight conversion
stylegan_nada_dir = '/content/User-Whole-Body-Generation/StyleGAN_Human'
stylegan_ada_dir  = '/content/User-Whole-Body-Generation/StyleGAN_Human'

source_model_type = 'ffhq'
source_model_download_path = {'ffhq': '1EM87UquaoQmk17Q8d5kYIAHqu0dkYqdT'}
model_names = {'ffhq': 'ffhq.pt'}

download_string = source_model_download_path[source_model_type]
file_name    = model_names[source_model_type]
pt_file_name = file_name.split('.')[0] + '.pt'

dataset_sizes = {'ffhq': 1024}

if not os.path.isfile(os.path.join(pretrained_model_dir, file_name)):
    print('Downloading chosen model...')
    if download_string.endswith('.pkl'):
        !wget $download_string -O $pretrained_model_dir/$file_name
    else:
        downloader.download_file(download_string, os.path.join(pretrained_model_dir, file_name))

if not os.path.isfile(os.path.join(pretrained_model_dir, pt_file_name)):
    print('Converting sg2 model — this may take a few minutes...')
    tf_path = next(filter(lambda x: 'tensorflow' in x, sys.path), '')
    py_path = f'{tf_path}:{stylegan_nada_dir}/ZSSGAN' if tf_path else f'{stylegan_nada_dir}/ZSSGAN'
    convert_script = os.path.join(stylegan_nada_dir, 'convert_weight.py')
    !PYTHONPATH=$py_path python $convert_script --repo $stylegan_ada_dir --gen $pretrained_model_dir/$file_name


### pSp Monkey-Patch: Shape-Compatible Decoder Loading

**Why this patch is needed:** PyTorch's `strict=False` only skips *missing* or *extra*  
keys — it still raises `RuntimeError` on *size mismatches*. The FFHQ checkpoint  
stores square noise tensors (e.g. `1024×1024`) but the portrait whole-body decoder  
expects half-width tensors (`1024×512`). This monkey-patch replaces `pSp.load_weights`  
before the class is instantiated, filtering the decoder state dict so only  
shape-compatible tensors are loaded — the 18 mismatched keys are silently skipped.


In [ ]:
# Monkey-patch pSp.load_weights BEFORE importing/instantiating it.
# Background: PyTorch strict=False only skips MISSING keys — shape mismatches
# still raise RuntimeError. The FFHQ checkpoint has square noise tensors
# (e.g. 1024×1024) but the portrait whole-body decoder expects half-width
# (1024×512). This patch filters the decoder state dict to only load tensors
# whose shapes match the current model.
from restyle.models.psp import pSp

def _patched_load_weights(self):
    if self.opts.checkpoint_path is not None:
        import torch as _torch
        ckpt = _torch.load(self.opts.checkpoint_path, map_location='cpu')
        self.encoder.load_state_dict(self._pSp__get_keys(ckpt, 'encoder'), strict=False)
        decoder_sd = self._pSp__get_keys(ckpt, 'decoder')
        model_sd   = self.decoder.state_dict()
        filtered   = {k: v for k, v in decoder_sd.items()
                      if k in model_sd and v.shape == model_sd[k].shape}
        skipped    = [k for k in decoder_sd if k not in filtered]
        if skipped:
            print(f'Decoder: skipped {len(skipped)} shape-mismatched key(s) (expected for portrait model)')
        self.decoder.load_state_dict(filtered, strict=False)
        self._pSp__load_latent_avg(ckpt)
    else:
        self._pSp__load_latent_avg(None)

pSp.load_weights = _patched_load_weights
print('pSp.load_weights patched — shape-mismatched decoder keys will be skipped')


pSp.load_weights patched — shape-mismatched decoder keys will be skipped


### Load ReStyle pSp Network

The monkey-patch injected two cells above replaces `load_weights` so that  
decoder tensors are filtered by shape before loading — this silently skips  
the 18 portrait-ratio noise buffers that don't match the FFHQ checkpoint.


### ReStyle Checkpoint Download

Downloads the pre-trained **pSp** (pixel2Style2pixel) encoder checkpoint from Google Drive. The download is guarded — skipped if the file already exists.  

An **e4e** (encoder4editing) alternative is commented out; it trades slight quality for better latent editability.

In [15]:
from restyle.utils.common import tensor2im
from restyle.models.psp import pSp
#from restyle.models.e4e import e4e
#pretrained_model_dir = '/content/User-Whole-Body-Generation/restyle'

filename_download = os.path.join(pretrained_model_dir, "restyle_psp_ffhq_encode.pt")
if not os.path.isfile(filename_download):
  downloader.download_file("1sw6I2lRIB0MpuJkpc8F5BJiSZrc0hjfE", os.path.join(pretrained_model_dir, "restyle_psp_ffhq_encode.pt"))

# filename_download = os.path.join(pretrained_model_dir, "restyle_e4e_ffhq_encode.pt")
# if not os.path.isfile(filename_download):
#   downloader.download_file("1e2oXVeBPXMQoUoC_4TNwAWpOPpSEhE_e", os.path.join(pretrained_model_dir, "restyle_e4e_ffhq_encode.pt"))

### Inference Utilities: `run_alignment()` & `get_avg_image()`

- `run_alignment` — detects the face, applies the 68-point predictor, crops and aligns
- `get_avg_image` — generates the average latent image used as the starting point for iterative encoding

**Input size:** `(256, 128)` — portrait ratio matching the whole-body decoder's noise buffers.  
Using `(256, 256)` causes a dimension mismatch when `run_on_batch` concatenates  
the input with the average image along the channel axis.


In [16]:
encoder_type = 'psp' 
restyle_experiment_args = {
    "model_path": os.path.join(pretrained_model_dir, f"restyle_{encoder_type}_ffhq_encode.pt"),
    "transform": transforms.Compose([
        transforms.Resize((256, 128)),
        transforms.ToTensor(),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])
}

model_path = restyle_experiment_args['model_path']
ckpt = torch.load(model_path, map_location='cpu')

opts = ckpt['opts']

opts['checkpoint_path'] = model_path
opts = Namespace(**opts)

restyle_net = pSp(opts) #(pSp if encoder_type == 'psp' else e4e)(opts)

restyle_net.eval()
restyle_net.cuda()
print('Model successfully loaded!')

Decoder: skipped 18 shape-mismatched key(s) (expected for portrait model)
Model successfully loaded!


### StyleGAN_Human on `sys.path`

Adds the `StyleGAN_Human` directory to `sys.path` so that `scripts/align_faces_parallel` (used inside `run_alignment`) resolves as a relative import regardless of the current working directory.

In [17]:
import sys
_sg_path = '/content/User-Whole-Body-Generation/StyleGAN_Human'
if _sg_path not in sys.path:
    sys.path.insert(0, _sg_path)

def run_alignment(image_path):
    print("inside run_alignment")
    print(image_path)
    import dlib
    from scripts.align_faces_parallel import align_face
    if not os.path.exists("shape_predictor_68_face_landmarks.dat"):
        print('Downloading files for aligning face image...')
        os.system('wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2')
        os.system('bzip2 -dk shape_predictor_68_face_landmarks.dat.bz2')
        print('Done.')
    predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
    print("before alight face")
    aligned_image = align_face(filepath=image_path, predictor=predictor) 
    print("Aligned image has shape: {}".format(aligned_image.size))
    return aligned_image 


def get_avg_image(net):
    avg_image = net(net.latent_avg.unsqueeze(0),
                    input_code=True,
                    randomize_noise=False,
                    return_latents=False,
                    average_code=True)[0]
    avg_image = avg_image.to('cuda').float().detach()
    return avg_image

### Working Directory

Set to `StyleGAN_Human` so that InsetGAN's relative imports (`insetgan1`,  
`generate1`, `legacy`, `utils/`) resolve correctly from that directory.


### `get_latent_code()` — ReStyle Face Inversion

Aligns the input image, runs it through the ReStyle pSp encoder for  
`n_iters_per_batch` iterations, and returns the final latent code array.  
This latent is passed directly to the InsetGAN optimiser.


### Inference Options & `get_latent_code()`

`opts.n_iters_per_batch = 5` — ReStyle runs 5 progressive encoding passes per image.  
`get_latent_code()` wraps the full alignment → encode → return-latents pipeline into a single callable used by both the API endpoints and the InsetGAN optimiser.

In [18]:
opts.n_iters_per_batch = 5
opts.resize_outputs = False  # generate outputs at full resolution
#image_path = "/content/User-Whole-Body-Generation/0F9A2782.jpg"  

from restyle.utils.inference_utils import run_on_batch

def get_latent_code (image_file_path) :

  original_image = Image.open(image_file_path).convert("RGB")
  input_image = run_alignment(image_file_path)
  display(input_image)
  
  img_transforms = restyle_experiment_args['transform']
  transformed_image = img_transforms(input_image)
  
  with torch.no_grad():
    avg_image = get_avg_image(restyle_net)
    print(transformed_image.unsqueeze(0).shape)
    result_batch, result_latents = run_on_batch(transformed_image.unsqueeze(0).cuda(), restyle_net, opts, avg_image)
    
  return result_latents   

In [19]:
os.chdir('/content/User-Whole-Body-Generation/StyleGAN_Human')

### Working Directory for InsetGAN

Moves back to `StyleGAN_Human` so that InsetGAN's relative imports  
(`insetgan1`, `generate1`, `legacy`, `utils/`) resolve correctly from  
the expected package root. This `chdir` is kept separate from the  
`get_latent_code()` definition above so it only takes effect immediately  
before the InsetGAN import cell.


### InsetGAN: `user_joint_optimisation()`

Takes face latent codes from ReStyle and runs the dual optimiser:

1. Samples a body from the body generator using `body_seed`
2. Detects the face region in the body image (dlib)
3. Inserts the face into the body and runs `joint_steps` optimisation iterations
4. Saves a PNG (final frame) and an MP4 (optimisation video) to `outputs/insetgan/`
5. Returns `[png_path, mp4_path]`

**Parameters**

| Param | Type | Description |
|---|---|---|
| `face_latent_codes` | list | Output from `get_latent_code()` |
| `image_name` | str | Source filename (used to name outputs) |
| `body_seed` | str | Integer seed for the body generator |
| `joint_steps` | str | Optimisation iterations (5–500, higher = better quality) |
| `trunc` | str | Truncation ψ (0–1, lower = more average, higher = more diverse) |


In [ ]:
import os, sys

# insetgan1.py imports using 'User_Whole_Body_Generation.StyleGAN_Human.*'
# but the cloned directory is 'User-Whole-Body-Generation' (hyphens).
# Python cannot import names with hyphens, so we create an underscore symlink.
if not os.path.exists('/content/User_Whole_Body_Generation'):
    os.symlink('/content/User-Whole-Body-Generation',
               '/content/User_Whole_Body_Generation')
    print('Symlink created: User_Whole_Body_Generation -> User-Whole-Body-Generation')
else:
    print('Symlink already exists.')

# Make /content the package root so Python can find User_Whole_Body_Generation.*
if '/content' not in sys.path:
    sys.path.insert(0, '/content')

print('sys.path[0]:', sys.path[0])


Symlink already exists.
sys.path[0]: /content/User-Whole-Body-Generation/StyleGAN_Human


In [20]:
#This is part of Main function code in Insetgan implementation. This is
#modified such that face seed implemenation is removed
#Instead face_latent_code is used as input 

from insetgan1 import InsetGAN
import torch
import torch.nn.functional as F
from tqdm import tqdm
from lpips import LPIPS
import numpy as np
from torch_utils.models import Generator as bodyGAN
from torch_utils.models_face import Generator as FaceGAN
import dlib
from utils.alignment import align_face_for_insetgan
from utils.util import visual,tensor_to_numpy, numpy_to_tensor
import legacy
import os
import click


def user_joint_optimisation( face_latent_codes , image_name ,  body_seed  , joint_steps ,trunc  ):

    face_network = "./pretrained_models/ffhq.pkl"
    body_network = "./pretrained_models/stylegan2_1024.pkl"
  
    body_seed_int = int(body_seed)
    joint_steps_int = int(joint_steps)
    trunc_float = float(trunc)
    
    image_filename_short = image_name.split('.')[0]
    body_seed = body_seed_int
    joint_steps= joint_steps_int 
    truncation_psi = trunc_float 
    outdir = 'outputs/insetgan'
    video = 1

    #Remove Directory functionality need to be implemented - Google Colab not supporting deletion of folder which is not empty
    #if os.path.exists(f'./{outdir}/{image_filename_short}_{body_seed:04d}') == True :
       # os.rmdir(f'./{outdir}/{image_filename_short}_{body_seed:04d}')
    

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    insgan = InsetGAN(body_network, face_network)
    os.makedirs(outdir, exist_ok=True)
    face_mean = insgan.face_generator.mean_latent(3000)
    face_w =  torch.Tensor(face_latent_codes[0][4]).cuda().unsqueeze(0) #.unsqueeze(1)
    print(face_w.shape)
    face_w = truncation_psi * face_w + (1-truncation_psi) * face_mean
    face_img, _ = insgan.face_generator([face_w], input_is_latent=True)
  
    body_z = np.random.RandomState(body_seed).randn(1, 512).astype(np.float32)
    body_mean = insgan.body_generator.mean_latent(3000)
    body_w = insgan.body_generator.get_latent(torch.from_numpy(body_z).to(device))  # [N, L, C]
    body_w = truncation_psi * body_w + (1-truncation_psi) * body_mean
    body_img, _ = insgan.body_generator([body_w], input_is_latent=True)

    _, body_crop, _ = insgan.detect_face_dlib(body_img)
    face_img = F.interpolate(face_img, size=(body_crop[3]-body_crop[1], body_crop[2]-body_crop[0]), mode='area')
    cp_body = body_img.clone()
    cp_body[:, :, body_crop[1]:body_crop[3], body_crop[0]:body_crop[2]] = face_img
    
    optim_face_w, optim_body_w, crop = insgan.dual_optimizer(
        face_w, 
        body_w,
        joint_steps=joint_steps,
        seed=f'{image_filename_short}_{body_seed:04d}',
        output_path=outdir,
        video=video
    )
    
    if video:
        ffmpeg_cmd = f"ffmpeg -hide_banner -loglevel error -i ./{outdir}/{image_filename_short}_{body_seed:04d}/%04d.jpg -c:v libx264 -vf fps=30 -pix_fmt yuv420p ./{outdir}/{image_filename_short}_{body_seed:04d}.mp4"
        os.system(ffmpeg_cmd)
  
    new_face_img, _ = insgan.face_generator([optim_face_w], input_is_latent=True)
    new_shape = crop[3] - crop[1], crop[2] - crop[0]
    new_face_img_crop = F.interpolate(new_face_img, size=new_shape, mode='area')
    seamless_body, _ = insgan.body_generator([optim_body_w], input_is_latent=True)
    seamless_body[:, :, crop[1]:crop[3], crop[0]:crop[2]] = new_face_img_crop
    temp = torch.cat([cp_body, seamless_body], dim=3)
    
    visual(temp, f"{outdir}/{image_filename_short}_{body_seed:04d}.png")
    
    path = '/content/User-Whole-Body-Generation/StyleGAN_Human/outputs/insetgan/'
    file_paths=['','']
    file_name_png = f'{image_filename_short}_{body_seed_int:04d}.png'
    file_paths[0] = os.path.join(path, file_name_png )
    file_name_mp4 = f'{image_filename_short}_{body_seed_int:04d}.mp4' 
    file_paths[1] = os.path.join(path, file_name_mp4 )

    return (file_paths)   


## Part 3 — API: Setup & Endpoints

A FastAPI application exposing the InsetGAN pipeline over HTTP.  
All CORS origins are permitted so any frontend can call the endpoints.


### Additional API Dependency

`python-multipart` is required by FastAPI to parse `multipart/form-data` file uploads (used by the `/join-optimisation-upload-endpoint`).

In [21]:
!pip install python-multipart #Check

### Video Streaming & HTML Routes

Defines the `/video` and `/html` routes.  
`video_path` is a module-level variable initialised to `None` and set by the joint-optimisation endpoints after a generation run.  
`/video` returns `404` until a generation has completed.

In [22]:
from pathlib import Path
from fastapi import Request, Response
from fastapi.templating import Jinja2Templates

templates = Jinja2Templates(directory='fastapi-video/templates')
CHUNK_SIZE = 1024 * 1024

# video_path is set by user_joint_optimisation() after a generation run.
# It starts as None so the /video endpoint returns 404 until a video exists.
video_path = None

@app.get('/html')
async def read_root(request: Request):
    return templates.TemplateResponse('index.htm', context={'request': request})

@app.get('/video')
async def video_endpoint():
    if video_path is None or not Path(video_path).exists():
        return Response(
            content='No video available yet. Run a generation first.',
            status_code=404,
            media_type='text/plain'
        )
    vp = Path(video_path)
    filesize = vp.stat().st_size
    data = vp.read_bytes()
    headers = {
        'Content-Range': f'bytes 0-{filesize-1}/{filesize}',
        'Accept-Ranges': 'bytes',
    }
    return Response(data, status_code=206, headers=headers, media_type='video/mp4')


### fastapi-video Template

Clones a minimal HTML template repo used by the `/html` preview endpoint.  
The `Jinja2Templates` directory points to its `templates/` folder.

In [23]:
if not os.path.exists('/content/fastapi-video'):
    !git clone https://github.com/aurthurm/fastapi-video.git
else:
    print('fastapi-video repo already exists, skipping clone.')


Cloning into 'fastapi-video'...
fatal: could not read Username for 'https://github.com': No such device or address


In [ ]:
# ⚠️  The ngrok token previously hardcoded here has been removed.
# It was committed to git history and must be REVOKED at:
# https://dashboard.ngrok.com/tunnels/authtokens
# Then generate a new token and paste it in the server startup cell below.


### API Routes — Joint Optimisation & Image Generation

Defines the main application and all routes.  
A **fresh `FastAPI()` instance** is always created so middleware can be added without raising `RuntimeError: Cannot add middleware after an application has started`.  

| Endpoint | Method | Description |
|---|---|---|
| `/join-optimisation-url-endpoint` | GET | Download face from `image_url`, run InsetGAN, return MP4 |
| `/join-optimisation-upload-endpoint` | POST | Upload face photo, run InsetGAN, return MP4 |
| `/generate_single_image` | GET | Generate a body image from a random `seed` |
| `/video` | GET | Stream the last generated MP4 |
| `/html` | GET | HTML preview page |
| `/docs` | GET | Auto-generated Swagger UI |

In [27]:
import threading, nest_asyncio, uvicorn
from pyngrok import ngrok

# 1. Set your ngrok auth token (get one free at https://dashboard.ngrok.com)
ngrok.set_auth_token('29LR1RD7sCbaYhliJQGaohZWYud_5A5g8831CiiBnNR2yvvra')

# 2. Open public tunnel
public_url = ngrok.connect(8000).public_url
print('Public URL:', public_url)
print('API docs: ', public_url + '/docs')

# 3. nest_asyncio allows uvicorn to run inside the Jupyter event loop
nest_asyncio.apply()

# 4. Start server in a daemon thread so this cell returns immediately
_server = threading.Thread(
    target=uvicorn.run,
    kwargs={'app': app, 'host': '0.0.0.0', 'port': 8000},
    daemon=True
)
_server.start()
print('Server running.')


Public URL: https://5582-34-125-197-82.ngrok-free.app
API docs:  https://5582-34-125-197-82.ngrok-free.app/docs
Server running.


INFO:     Started server process [591]
INFO:     Waiting for application startup.
INFO:     Application startup complete.


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


### Server Startup — uvicorn + pyngrok

1. Replace `YOUR_NGROK_TOKEN` with your token from [dashboard.ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)  
2. An ngrok tunnel is opened on port 8000  
3. `nest_asyncio.apply()` allows uvicorn to run inside the Jupyter event loop  
4. The server starts in a **daemon thread** so this cell returns immediately  

> **Note:** re-run the API routes cell first if you re-run the server — the `app` object must be freshly created before `uvicorn.run()` is called.

In [ ]:
import urllib.request, shutil
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware

# Always create a fresh app so middleware can be added cleanly.
# Re-running this cell rebuilds all routes from scratch.
app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

# ── Joint optimisation via URL ────────────────────────────────────────────
@app.get('/join-optimisation-url-endpoint')
async def joint_optimisation_endpoint(
    image_url: str,
    body_seed: str = '20345',
    joint_steps: str = '5',
    trunc: str = '1'
):
    global video_path
    colab_image_path = '/content/User-Whole-Body-Generation'
    local_img = os.path.join(colab_image_path, 'up_image.jpg')
    try:
        urllib.request.urlretrieve(image_url, 'test.jpg')
    except Exception as e:
        return {'error': f'Failed to download image: {e}'}
    Image.open('test.jpg').save(local_img)
    face_latent_codes = get_latent_code(local_img)
    filepaths = user_joint_optimisation(
        face_latent_codes, 'up_image.jpg', body_seed, joint_steps, trunc)
    video_path = filepaths[1]
    return FileResponse(
        path=filepaths[1],
        headers={'content-disposition': 'attachment', 'filename': 'result.mp4'},
        media_type='video/mp4',
        filename='result.mp4'
    )

# ── Joint optimisation via file upload ───────────────────────────────────
@app.post('/join-optimisation-upload-endpoint')
async def joint_optimisation_upload(
    file: UploadFile = File(...),
    body_seed: str = '20345',
    joint_steps: str = '5',
    trunc: str = '1'
):
    global video_path
    colab_image_path = '/content/User-Whole-Body-Generation'
    local_img = os.path.join(colab_image_path, 'up_image.jpg')
    with open(local_img, 'wb') as f:
        shutil.copyfileobj(file.file, f)
    face_latent_codes = get_latent_code(local_img)
    filepaths = user_joint_optimisation(
        face_latent_codes, 'up_image.jpg', body_seed, joint_steps, trunc)
    video_path = filepaths[1]
    return FileResponse(
        path=filepaths[1],
        headers={'content-disposition': 'attachment', 'filename': 'result.mp4'},
        media_type='video/mp4',
        filename='result.mp4'
    )

# ── Single image from random seed ────────────────────────────────────────
@app.get('/generate_single_image')
def generate_single_image_endpoint(seed: str = '12', trunc: float = 0.5):
    output_path = '/content/User-Whole-Body-Generation/StyleGAN_Human/outputs/stylegan2_1024'
    generate_images(
        network_pkl='pretrained_models/stylegan2_1024.pkl',
        seeds=legacy.num_range(seed),
        truncation_psi=trunc,
        outdir=output_path,
        noise_mode='const',
        version=2
    )
    for seed_val in legacy.num_range(seed):
        file_name = f'seed{seed_val:04d}.png'
        file_path = os.path.join(output_path, file_name)
        if os.path.exists(file_path):
            return FileResponse(file_path, media_type='image/png', filename=file_name)
    return {'error': 'File not found'}

print('App and routes registered.')


App and routes registered.


In [ ]:
# Server is started in cell [47] above via uvicorn + pyngrok.
# This cell is kept as a placeholder — nothing to run here.
print('Server is managed by the cell above.')


Server is managed by the cell above.


## Testing

Commented-out test code below shows the full local test flow.  
Uncomment and adapt `image_path_test` to run inference end-to-end without the API.


In [ ]:
#Code to test locally

# image_path_test = "/content/User-Whole-Body-Generation/0F9A2782.jpg"

# def user_joint_optimisation_test( body_seed = '20345' , joint_steps= '500' ,trunc ='1') :
  
#       urllib.request.urlretrieve('https://firebasestorage.googleapis.com/v0/b/whole-body-gan-demo.appspot.com/o/image.jpg?alt=media','test.jpg')  
#       #upload_file_path = 
#       uploaded_image = Image.open('test.jpg')
#       colab_image_path = '/content/User-Whole-Body-Generation'
#       print("Shutil copy")
#       #new_image = shutil.copy(uploaded_image,colab_image_path)
#       #print("after Shutil copy")
#       uploaded_image.save(os.path.join(colab_image_path,'up_image.jpg')) 
#       face_latent_codes = get_latent_code (os.path.join(colab_image_path,'up_image.jpg'))
      
#       #face_latent_codes = get_latent_code (image_filepath)
#       filepaths = user_joint_optimisation( face_latent_codes, 'up_image.jpg' , body_seed, joint_steps ,trunc )

#       print(filepaths[1])

# user_joint_optimisation_test() 
